# College Earnings Prediction Using Machine Learning

**Author:** Carson Bell  
**Date:** April 19, 2026

This project predicts post-graduation earnings using college-level data. Two machine learning models (Random Forest and Neural Network) are implemented, optimized, and compared to identify key drivers of earnings and provide actionable insights.

## 1. Problem Statement & Dataset

The goal of this project is to predict median post-graduation earnings using institutional characteristics such as tuition, debt, and admissions metrics. This helps evaluate the financial return on investment (ROI) of attending different colleges.

**Dataset:** U.S. College Scorecard  
https://collegescorecard.ed.gov/data/

The dataset contains thousands of colleges and hundreds of features related to cost, admissions, and outcomes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/raw/college_data.csv")

print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/college_data.csv'

In [ ]:
df.describe()
df.isna().sum().sort_values(ascending=False).head(10)

The dataset contains missing values across several columns, particularly in earnings-related fields. Careful feature selection is required to ensure a usable target variable.

In [ ]:
target_col = "MD_EARN_WNE_INDEP1_P11"

features = [
    'TUITIONFEE_OUT',
    'DEBT_MDN',
    'ADM_RATE_ALL',
    'SAT_AVG_ALL'
]

df_model = df[features + [target_col]].copy()

In [ ]:
df_model = df_model.replace(['NULL', 'PrivacySuppressed', ''], np.nan)
df_model = df_model.apply(pd.to_numeric, errors='coerce')

df_model = df_model.dropna(subset=[target_col])
df_model = df_model.fillna(df_model.mean())

print(df_model.shape)

In [ ]:
df_model['cost_to_earnings'] = df_model['TUITIONFEE_OUT'] / df_model[target_col]
df_model['debt_to_earnings'] = df_model['DEBT_MDN'] / df_model[target_col]
df_model['selectivity_score'] = df_model['SAT_AVG_ALL'] * (1 - df_model['ADM_RATE_ALL'])
df_model['cost_plus_debt'] = df_model['TUITIONFEE_OUT'] + df_model['DEBT_MDN']
df_model['sat_per_cost'] = df_model['SAT_AVG_ALL'] / df_model['TUITIONFEE_OUT']

These engineered features capture:
- Return on investment (cost vs earnings)
- Financial burden (debt vs earnings)
- Institutional competitiveness (SAT vs admission rate)
- Combined financial cost
- Academic quality per dollar

In [ ]:
X = df_model.drop(target_col, axis=1)
y = df_model[target_col]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

nn = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42
)

nn.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# RF
rf_pred = rf.predict(X_test)

# NN
nn_pred = nn.predict(X_test_scaled)

rf_rmse = mean_squared_error(y_test, rf_pred, squared=False)
nn_rmse = mean_squared_error(y_test, nn_pred, squared=False)

rf_r2 = r2_score(y_test, rf_pred)
nn_r2 = r2_score(y_test, nn_pred)

print("RF RMSE:", rf_rmse)
print("NN RMSE:", nn_rmse)
print("RF R2:", rf_r2)
print("NN R2:", nn_r2)

In [ ]:
plt.scatter(y_test, rf_pred)
plt.title("Random Forest Predictions")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

In [ ]:
plt.scatter(y_test, nn_pred)
plt.title("Neural Network Predictions")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df

In [ ]:
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.title("Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Neural Network'],
    'RMSE': [rf_rmse, nn_rmse],
    'R2': [rf_r2, nn_r2]
})

comparison

The Random Forest model outperformed the Neural Network in both RMSE and R². This is likely due to the structured nature of the dataset, where tree-based models capture nonlinear relationships more effectively without requiring extensive scaling.

The Neural Network required feature scaling and showed slightly lower performance, suggesting that additional tuning or more data may be needed.

Given the results, the Random Forest model is the preferred choice due to its higher accuracy and interpretability.

## Ethical Analysis & Responsible Deployment

Potential bias may arise from historical inequalities in higher education, such as differences in access to resources across institutions. Schools serving underrepresented populations may systematically show lower earnings outcomes, which could reinforce inequities if used improperly.

Incorrect predictions could mislead students into avoiding certain institutions or overvaluing others. This could disproportionately impact low-income or minority students.

To mitigate these risks, the model should be used as a decision-support tool rather than a sole decision-maker. Additional fairness analysis and demographic controls should be incorporated before deployment.

## Business Recommendations & Deployment

Students and families should use these predictions to evaluate the financial return of college choices. Schools with strong earnings relative to cost may provide better ROI.

The model should be deployed as an advisory tool within a college planning platform, with transparency about its limitations.

Limitations include missing variables (such as major choice) and potential bias in earnings data. The model should not be used as the sole determinant of college value.

Future improvements include:
- Additional feature engineering
- Hyperparameter tuning
- Incorporating more variables (region, major)
- Testing advanced models (XGBoost)

In [ ]:
git add .
git commit -m "final project complete"
git push